# Car Damage Detection — YOLO11s

This notebook documents dataset retrieval, inspection, model training, prediction, and held-out test evaluation for the vehicle-damage detector.

**Security:** add a `ROBOFLOW_API_KEY` secret in Google Colab (left sidebar → Secrets) and enable notebook access. Do not paste credentials into notebook cells.


In [ ]:
!pip install ultralytics

In [ ]:
!pip install roboflow

In [ ]:
from roboflow import Roboflow
from google.colab import userdata


## Download the dataset


In [ ]:
from roboflow import Roboflow
from google.colab import userdata

api_key = userdata.get("ROBOFLOW_API_KEY")
if not api_key:
    raise RuntimeError("Add ROBOFLOW_API_KEY in Colab Secrets and enable notebook access.")

rf = Roboflow(api_key=api_key)
project = rf.workspace("fuad-nabiyev").project("car-damage-zxk33-ucsnt")
version = project.version(1)
dataset = version.download("yolov11")


In [ ]:
from pathlib import Path

In [ ]:
DATASET_PATH = Path('/content/car-damage-1')

print("Dataset exists:", DATASET_PATH.exists())
print("\nFiles:")
for item in DATASET_PATH.iterdir():
  print(item)

In [ ]:
yaml_path = DATASET_PATH / "data.yaml"
print("\n===== data.yaml =====\n")
print(yaml_path.read_text())

In [ ]:
from pathlib import Path
for split in ["train", "valid", "test"]:
  image_dir = DATASET_PATH / split / "images"

  if image_dir.exists():
    image = list(image_dir.glob("*"))
    print(f"{split}: {len(image)} images")
  else:
    print(f"{split}: folder not found")

In [ ]:
import yaml

with open(DATASET_PATH / "data.yaml", 'r') as f:
  data = yaml.safe_load(f)

print("Number of classes:", data["nc"])
print("\nClasses:")

for i, name in enumerate(data["names"]):
  print(f"{i}: {name}")

In [ ]:
DATASET_PATH = Path("/content/car-damage-1")
for split in ["train", "valid", "test"]:
  image_dir = DATASET_PATH / split / "images"
  label_dir = DATASET_PATH / split / "labels"

  images = list(image_dir.glob("*"))
  labels = list(label_dir.glob("*.txt"))

  print(f"{split.upper()}")
  print(f"  Images: {len(images)}")
  print(f"  Lavels: {len(labels)}")
  print()

In [ ]:
import torch

In [ ]:
if torch.cuda.is_available():
  device = "cuda"
elif torch.backends.mps.is_available():
  device = "mps"
else:
  device ="cpu"

In [ ]:
print("Using device:", device)

In [ ]:
from ultralytics import YOLO

## Load the pretrained YOLO11s model


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")


## Train the model


In [ ]:
results = model.train(
    data="/content/car-damage-1/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device="cuda",
    patience=10,
    seed=42,
    workers=2,
    project="content/car_damage_training",
    name="yolo11s_car_damage",
    pretrained=True
)

In [ ]:
import os

test_images = "/content/car-damage-1/test/images"

print(os.listdir(test_images)[:10])

In [ ]:
results = model.predict(
    source="/content/car-damage-1/test/images/12_jpeg_jpg.rf.2067268dc6cc6558a1b7d37a01a5bcc4.jpg",
    conf=0.25,
    save=True
)

## Evaluate on the held-out test split


In [ ]:
metrics = model.val(
    data="/content/car-damage-1/data.yaml",
    split="test"
)

In [ ]:
model.val(
    data="/content/car-damage-1/data.yaml",
    split="val",
    plots=True
)

In [ ]:
results = model.predict(
    source="/content/car-damage-1/test/images",
    conf=0.25,
    save=True
)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
img = Image.open(
    "/content/runs/detect/content/car_damage_training/yolo11s_car_damage/results.png"
)

plt.figure(figsize=(15, 10))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

folder = "/content/runs/detect/predict"

images = [
    f for f in os.listdir(folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

for image_name in images[:6]:
    img = Image.open(os.path.join(folder, image_name))

    plt.figure(figsize=(10, 7))
    plt.imshow(img)
    plt.title(image_name)
    plt.axis("off")
    plt.show()

In [ ]:
from google.colab import files

In [ ]:
files.download(
    "/content/runs/detect/content/car_damage_training/yolo11s_car_damage/weights/best.pt"
)

## Recorded test results

The saved notebook output reported the following aggregate metrics on 1,304 test images and 1,916 labeled instances:

| Metric | Test result |
|---|---:|
| Precision | 0.904 |
| Recall | 0.853 |
| mAP@50 | 0.847 |
| mAP@50–95 | 0.751 |

These are the results recorded in the original run, not a guarantee that a fresh run will reproduce identical values. Some classes have very few test examples, so per-class scores for rare classes are unstable.
